In [ ]:
# Load data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

In [ ]:
df = pd.read_csv("../data/processed/cleaned_default_data.csv")

In [ ]:
df.columns = df.columns.str.strip()

target = "default payment next month"
feature_cols = [col for col in df.columns if col != target]

X = df[feature_cols].values
y = df[target].values

# Scale features (same pattern)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=feature_cols)

# ---- Baseline MLE Logistic (same spirit as likelihood_mle.ipynb) ----
X_sm = sm.add_constant(X_scaled_df)
logit_model = sm.Logit(y, X_sm)
mle_result = logit_model.fit(method="newton", maxiter=200, disp=False)

print("Baseline MLE (unpenalized) fitted.")
print(mle_result.summary())

# ---- Train/Test for prediction comparison ----
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, stratify=y, random_state=42
)

# Baseline sklearn logistic (unpenalized approximation via very large C)
baseline = LogisticRegression(
    penalty="l2", C=1e6, solver="lbfgs", max_iter=5000, random_state=42
)
baseline.fit(X_train, y_train)

# Ridge (L2) with CV
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
Cs = np.logspace(-3, 2, 20)

ridge_cv = LogisticRegressionCV(
    Cs=Cs, cv=cv, penalty="l2", solver="saga",
    scoring="roc_auc", max_iter=5000, random_state=42, n_jobs=-1
)
ridge_cv.fit(X_train, y_train)

# Lasso (L1) with CV
lasso_cv = LogisticRegressionCV(
    Cs=Cs, cv=cv, penalty="l1", solver="saga",
    scoring="roc_auc", max_iter=5000, random_state=42, n_jobs=-1
)
lasso_cv.fit(X_train, y_train)

def eval_model(name, model, X_te, y_te):
    pred = model.predict(X_te)
    prob = model.predict_proba(X_te)[:, 1]
    return {
        "model": name,
        "accuracy": accuracy_score(y_te, pred),
        "precision": precision_score(y_te, pred, zero_division=0),
        "recall": recall_score(y_te, pred, zero_division=0),
        "f1": f1_score(y_te, pred, zero_division=0),
        "roc_auc": roc_auc_score(y_te, prob),
    }

results = pd.DataFrame([
    eval_model("Unpenalized Logistic", baseline, X_test, y_test),
    eval_model("Ridge (L2) CV", ridge_cv, X_test, y_test),
    eval_model("Lasso (L1) CV", lasso_cv, X_test, y_test),
]).sort_values("roc_auc", ascending=False)

print("\nModel comparison:")
print(results)

# ---- Coefficient shrinkage comparison ----
coef_compare = pd.DataFrame({
    "Baseline": baseline.coef_.ravel(),
    "Ridge_L2": ridge_cv.coef_.ravel(),
    "Lasso_L1": lasso_cv.coef_.ravel()
}, index=feature_cols)

coef_compare["abs_mean"] = coef_compare.abs().mean(axis=1)
coef_compare = coef_compare.sort_values("abs_mean", ascending=False).drop(columns=["abs_mean"])

print("\nTop 12 features by average absolute coefficient:")
print(coef_compare.head(12))

# Plot top coefficients
top_n = 12
coef_compare.head(top_n).plot(kind="bar", figsize=(12, 6))
plt.title("Coefficient Shrinkage: Baseline vs Ridge vs Lasso")
plt.ylabel("Coefficient value")
plt.xlabel("Feature")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

# Optional: sparsity check for Lasso
n_near_zero = (np.abs(coef_compare["Lasso_L1"]) < 1e-3).sum()
print(f"\nLasso near-zero coefficients (<1e-3): {n_near_zero} / {len(feature_cols)}")

NameError: name 'df' is not defined